In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, recall_score, precision_score
import matplotlib.pyplot as plt


In [2]:
# 1. Load Data
tickets = pd.read_csv("../Data/Raw/Customer_Service.csv")

# 2. Clean & Format
tickets["Open_Date"] = pd.to_datetime(tickets["Open_Date"])

# 3. Create the Target Variable: High_Risk_Outcome
# A ticket is High Risk if it takes >48 hours, is unresolved, OR has CSAT <= 3
tickets["High_Risk_Outcome"] = (
    (tickets["Resolution_Hours"]> 48) | (tickets["Resolved"] == "No") | (tickets["CSAT_1_to_5"] <= 3 )
    ).astype(int)
print("Target Distribution:")
print(tickets["High_Risk_Outcome"].value_counts(normalize=True))

# 4. Feature Engineering
tickets["Day_of_Week"] = tickets["Open_Date"].dt.day_name()
tickets["Month"] = tickets["Open_Date"].dt.month


# 5. Select Features for the Model
# We use features known AT or shortly after ticket creation
features = [
    "First_Response_Minutes",
    "Channel",
    "Issue_Type",
    "Day_of_Week",
    "Month"
]
target = "High_Risk_Outcome"

X = tickets[features]
y = tickets[target]


print(f"\nFeatures selected: {features}")
print(f"X shape: {X.shape}, y shape: {y.shape}")

Target Distribution:
High_Risk_Outcome
0    0.729538
1    0.270462
Name: proportion, dtype: float64

Features selected: ['First_Response_Minutes', 'Channel', 'Issue_Type', 'Day_of_Week', 'Month']
X shape: (6500, 5), y shape: (6500,)


In [3]:
# 1. Train/Test Split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 1. Define our features
categorical_features = ["Channel", "Issue_Type", "Day_of_Week", "Month"]
numeric_features = ["First_Response_Minutes"]


# 2. Define the Preprocessor 
# This encodes the text columns and automatically keeps the numeric columns
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features )
    ]
)


# 3. Define the models we want to compare

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, class_weight="balanced")
}

results = []


# 4. Loop through each model, train it, and evaluate it
for name, model in models.items():
    
    # Create the pipeline
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    # Train the model
    pipeline.fit(X_train, y_train)

    # Make predictions
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    # Calculate Business Metrics
    roc_auc = roc_auc_score(y_test, y_prob)
    recall = recall_score(y_test, y_pred) 
    precision = precision_score(y_test, y_pred) 

    # Save results
    results.append({
        "Model": name,
        "ROC-AUC": round(roc_auc, 3),
        "Recall (High Risk)": round(recall, 3),
        "Precision": round(precision, 3)
    })

# 5. Display the comparison table
results_df = pd.DataFrame(results)
print("\n--- MODEL COMPARISON ---")
print(results_df.to_string(index=False))



--- MODEL COMPARISON ---
              Model  ROC-AUC  Recall (High Risk)  Precision
Logistic Regression    0.529               0.506      0.282
      Decision Tree    0.515               0.202      0.298
      Random Forest    0.507               0.409      0.293


In [4]:
# 1. Change the Target to Extreme Anger (CSAT == 1)
tickets["Extreme_Anger"] = (tickets["CSAT_1_to_5"] == 1).astype(int)

print("New Target Distribution:")
print(tickets["Extreme_Anger"].value_counts(normalize=True).round(3))

# 2. Redefine X and y
X = tickets[features]
y = tickets["Extreme_Anger"]

# 3. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Re-run the Model Comparison
results_extreme = []

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    roc_auc = roc_auc_score(y_test, y_prob)
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)

    results_extreme.append({
        "Model": name,
        "ROC-AUC": round(roc_auc, 3),
        "Recall": round(recall, 3),
        "Precision": round(precision, 3)
    })

print("\n--- EXTREME ANGER MODEL COMPARISON ---")
print(pd.DataFrame(results_extreme).to_string(index=False))

New Target Distribution:
Extreme_Anger
0    0.948
1    0.052
Name: proportion, dtype: float64

--- EXTREME ANGER MODEL COMPARISON ---
              Model  ROC-AUC  Recall  Precision
Logistic Regression    0.529   0.552      0.062
      Decision Tree    0.500   0.448      0.052
      Random Forest    0.536   0.209      0.074


In [5]:
import pandas as pd
from pathlib import Path

# 1. Point to your raw data directory
DATA_DIR = Path("../Data/Raw")  # Update path if running outside your Notebooks folder

print("🔍 SCANNING CSV HEADERS IN DATASET:\n" + "=" * 60)

# 2. Iterate through all CSV files in alphabetical order
for file_path in sorted(DATA_DIR.glob("*.csv")):
    try:
        # nrows=0 reads only headers; sep=None auto-detects tab vs comma delimiters
        df = pd.read_csv(file_path, nrows=0, sep=None, engine="python")
        cols = df.columns.tolist()
        
        print(f"📄 {file_path.name} ({len(cols)} columns)")
        print(f"   {cols}\n")
    except Exception as e:
        print(f"❌ Could not read {file_path.name}: {e}\n")

print("=" * 60)

🔍 SCANNING CSV HEADERS IN DATASET:
📄 Business_Units.csv (5 columns)
   ['\ufeffBusiness_Unit', 'Function', 'Public_Capacity', 'Source_URL', 'Verification_Class']

📄 Case_Briefs.csv (4 columns)
   ['\ufeffCard_No', 'Title', 'Business_Problem', 'Expected_Deliverable']

📄 Customer_Service.csv (9 columns)
   ['\ufeffTicket_ID', 'Open_Date', 'Channel', 'Issue_Type', 'First_Response_Minutes', 'Resolution_Hours', 'Resolved', 'CSAT_1_to_5', 'Data_Class']

📄 Customers.csv (9 columns)
   ['\ufeffCustomer_ID', 'Customer_Type', 'Governorate', 'Age_Band', 'Gender', 'Join_Date', 'Acquisition_Channel', 'Consent_Status', 'Data_Class']

📄 Data_Dictionary.csv (4 columns)
   ['\ufeffTable', 'Grain', 'Purpose', 'Real_vs_Modeled']

📄 Group_Operations.csv (7 columns)
   ['\ufeffMonth', 'Business_Unit', 'Primary_KPI', 'Volume', 'Operational_Efficiency_Pct', 'Open_Issues_Count', 'Data_Class']

📄 Inventory_History.csv (19 columns)
   ['\ufeffSnapshot_Month', 'Location_ID', 'Location_Name', 'Product_ID', 'SKU',